# Feature Engineering

In [ ]:
#importando bibliotecas
import pandas as pd
import plotly.express as px

import pandas as pd
from sklearn.cluster import KMeans
from kneed import KneeLocator

from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OrdinalEncoder

pd.set_option('display.max_columns', None)

In [2]:
def find_best_k(df, variables, max_k=10):
    k_range = range(1, max_k + 1)
    sse = []
    data = df[variables]
    
    for i in k_range:
        kmeans = KMeans(n_clusters=i, random_state=0, n_init='auto')
        kmeans.fit(data)
        sse.append(kmeans.inertia_)
    
    kl = KneeLocator(k_range, sse, curve='convex', direction='decreasing')
    best_k = kl.elbow
    
    return best_k

In [3]:
#importando dados
df = pd.read_pickle('../data/curated/data.pkl')

In [ ]:
#filtrando dados
#df = df[df['ORIGIN_STATE'] == 'NY']

In [5]:
df.columns

Index(['YEAR', 'MONTH', 'DAY', 'DAY_OF_WEEK', 'AIRLINE', 'FLIGHT_NUMBER',
       'TAIL_NUMBER', 'ORIGIN_AIRPORT', 'DESTINATION_AIRPORT',
       'SCHEDULED_DEPARTURE', 'DEPARTURE_TIME', 'DEPARTURE_DELAY', 'TAXI_OUT',
       'WHEELS_OFF', 'SCHEDULED_TIME', 'ELAPSED_TIME', 'AIR_TIME', 'DISTANCE',
       'WHEELS_ON', 'TAXI_IN', 'SCHEDULED_ARRIVAL', 'ARRIVAL_TIME',
       'ARRIVAL_DELAY', 'DIVERTED', 'CANCELLED', 'AIR_SYSTEM_DELAY',
       'SECURITY_DELAY', 'AIRLINE_DELAY', 'LATE_AIRCRAFT_DELAY',
       'WEATHER_DELAY', 'AIRLINE_NAME', 'ORIGIN_AIRPORT_NAME', 'ORIGIN_CITY',
       'ORIGIN_STATE', 'ORIGIN_COUNTRY', 'ORIGIN_LATITUDE', 'ORIGIN_LONGITUDE',
       'DEST_AIRPORT_NAME', 'DEST_CITY', 'DEST_STATE', 'DEST_COUNTRY',
       'DEST_LATITUDE', 'DEST_LONGITUDE', 'STATION_ORIGIN', 'DATE',
       'ORIGIN_AWND', 'ORIGIN_PRCP', 'ORIGIN_SNOW', 'ORIGIN_SNWD',
       'ORIGIN_TMAX', 'ORIGIN_TMIN', 'ORIGIN_WSF2', 'ORIGIN_WT01',
       'IS_DELAYED'],
      dtype='str')

Definindo perfil do aeroporto de origem

In [6]:
df_origin_airports_profile = df.groupby('ORIGIN_AIRPORT').agg({
    'IS_DELAYED': ['mean', 'count']
}).reset_index()
df_origin_airports_profile.columns = ['ORIGIN_AIRPORT', 'DELAY_RATE', 'FLIGHT_VOLUME']

n_clusters = find_best_k(df_origin_airports_profile, ['DELAY_RATE', 'FLIGHT_VOLUME'])
kmeans = KMeans(n_clusters=n_clusters, random_state=42)
df_origin_airports_profile['ORIGIN_AIRPORT_PROFILE'] = kmeans.fit_predict(df_origin_airports_profile[['DELAY_RATE', 'FLIGHT_VOLUME']])

In [7]:
fig = px.scatter(
    df_origin_airports_profile, 
    x='FLIGHT_VOLUME', 
    y='DELAY_RATE',
    color='ORIGIN_AIRPORT_PROFILE',
    hover_name='ORIGIN_AIRPORT',
    title='Agrupamento de Aeroportos: Volume vs. Taxa de Atraso',
    labels={'FLIGHT_VOLUME': 'Volume Total de Voos', 'DELAY_RATE': 'Taxa de Atraso (0 a 1)'},
    template='plotly_white',
    color_discrete_sequence=px.colors.qualitative.Safe
)

fig.show()

A análise de agrupamento revelou que o volume de operações é o principal fator de diferenciação entre os aeroportos. Enquanto grandes centros (hubs) mantêm uma taxa de atraso constante entre 15% e 25%, aeroportos de pequeno porte apresentam alta volatilidade, sendo mais suscetíveis a picos de atraso superiores a 40%, provavelmente devido à menor resiliência operacional.

Definindo perfil do aeroporto de destino

In [8]:
df_dest_airports_profile = df.groupby('DESTINATION_AIRPORT').agg({
    'IS_DELAYED': ['mean', 'count']
}).reset_index()
df_dest_airports_profile.columns = ['DESTINATION_AIRPORT', 'DELAY_RATE', 'FLIGHT_VOLUME']

n_clusters = find_best_k(df_dest_airports_profile, ['DELAY_RATE', 'FLIGHT_VOLUME'])
kmeans = KMeans(n_clusters=n_clusters, random_state=42)
df_dest_airports_profile['DESTINATION_AIRPORT_PROFILE'] = kmeans.fit_predict(df_dest_airports_profile[['DELAY_RATE', 'FLIGHT_VOLUME']])

In [9]:
fig = px.scatter(
    df_dest_airports_profile, 
    x='FLIGHT_VOLUME', 
    y='DELAY_RATE',
    color='DESTINATION_AIRPORT_PROFILE',
    hover_name='DESTINATION_AIRPORT',
    title='Agrupamento de Aeroportos: Volume vs. Taxa de Atraso',
    labels={'FLIGHT_VOLUME': 'Volume Total de Voos', 'DELAY_RATE': 'Taxa de Atraso (0 a 1)'},
    template='plotly_white',
    color_discrete_sequence=px.colors.qualitative.Safe
)

fig.show()

Definindo perfil da rota

In [10]:
df['ROUTE'] = df['ORIGIN_AIRPORT'] + '_' + df['DESTINATION_AIRPORT']

df_route_profile = df.groupby('ROUTE').agg({
    'IS_DELAYED': ['mean', 'count']
}).reset_index()
df_route_profile.columns = ['ROUTE', 'DELAY_RATE', 'FLIGHT_VOLUME']

n_clusters = find_best_k(df_route_profile, ['DELAY_RATE', 'FLIGHT_VOLUME'])
kmeans = KMeans(n_clusters=n_clusters, random_state=42)
df_route_profile['ROUTE_PROFILE'] = kmeans.fit_predict(df_route_profile[['DELAY_RATE', 'FLIGHT_VOLUME']])

In [11]:
fig = px.scatter(
    df_route_profile, 
    x='FLIGHT_VOLUME', 
    y='DELAY_RATE',
    color='ROUTE_PROFILE',
    hover_name='ROUTE',
    title='Agrupamento de Aeroportos: Volume vs. Taxa de Atraso',
    labels={'FLIGHT_VOLUME': 'Volume Total de Voos', 'DELAY_RATE': 'Taxa de Atraso (0 a 1)'},
    template='plotly_white',
    color_discrete_sequence=px.colors.qualitative.Safe
)

fig.show()

Definindo perfil da companhia aérea

In [12]:
df_airline_profile = df.groupby('AIRLINE').agg({
    'IS_DELAYED': ['mean', 'count']
}).reset_index()
df_airline_profile.columns = ['AIRLINE', 'DELAY_RATE', 'FLIGHT_VOLUME']

n_clusters = find_best_k(df_airline_profile, ['DELAY_RATE', 'FLIGHT_VOLUME'])
kmeans = KMeans(n_clusters=n_clusters, random_state=42)
df_airline_profile['AIRLINE_PROFILE'] = kmeans.fit_predict(df_airline_profile[['DELAY_RATE', 'FLIGHT_VOLUME']])

In [13]:
fig = px.scatter(
    df_airline_profile, 
    x='FLIGHT_VOLUME', 
    y='DELAY_RATE',
    color='AIRLINE_PROFILE',
    hover_name='AIRLINE',
    title='Agrupamento de Aeroportos: Volume vs. Taxa de Atraso',
    labels={'FLIGHT_VOLUME': 'Volume Total de Voos', 'DELAY_RATE': 'Taxa de Atraso (0 a 1)'},
    template='plotly_white',
    color_discrete_sequence=px.colors.qualitative.Safe
)

fig.show()

Obtendo período do dia

In [14]:
def get_time_of_day(raw):
    hour = raw // 100
    if 0 <= hour < 6:
        return 'OVERNIGHT'
    elif 6 <= hour < 12:
        return 'MORNING'
    elif 12 <= hour < 18:
        return 'AFTERNOON'
    else:
        return 'EVENING'

df['TIME_OF_DAY'] = df['SCHEDULED_DEPARTURE'].apply(get_time_of_day)

Obtendo estação do ano

In [15]:
seasons = {
    12: 'SUMMER', 1: 'SUMMER', 2: 'SUMMER',
    3: 'AUTUMN', 4: 'AUTUMN', 5: 'AUTUMN',
    6: 'WINTER', 7: 'WINTER', 8: 'WINTER',
    9: 'SPRING', 10: 'SPRING', 11: 'SPRING'
}

df['SEASON'] = df['MONTH'].map(seasons)

Obtendo horário programado do voo

In [16]:
df.SCHEDULED_ARRIVAL

0          430
1          750
2          806
3          805
4          320
          ... 
5819068    603
5819069    741
5819070    805
5819073    546
5819074    819
Name: SCHEDULED_ARRIVAL, Length: 4014615, dtype: int64

In [17]:
df['SCHEDULED_DEPARTURE_HOUR'] = df['SCHEDULED_DEPARTURE'] // 100
df['SCHEDULED_ARRIVAL_HOUR'] = df['SCHEDULED_ARRIVAL'] // 100

Mergeando os dados

In [18]:
df = df.merge(df_origin_airports_profile[['ORIGIN_AIRPORT', 'ORIGIN_AIRPORT_PROFILE']], on='ORIGIN_AIRPORT', how='left')
df = df.merge(df_dest_airports_profile[['DESTINATION_AIRPORT', 'DESTINATION_AIRPORT_PROFILE']], on='DESTINATION_AIRPORT', how='left')
df = df.merge(df_airline_profile[['AIRLINE', 'AIRLINE_PROFILE']], on='AIRLINE', how='left')
df = df.merge(df_route_profile[['ROUTE', 'ROUTE_PROFILE']], on='ROUTE', how='left')

In [19]:
df.columns

Index(['YEAR', 'MONTH', 'DAY', 'DAY_OF_WEEK', 'AIRLINE', 'FLIGHT_NUMBER',
       'TAIL_NUMBER', 'ORIGIN_AIRPORT', 'DESTINATION_AIRPORT',
       'SCHEDULED_DEPARTURE', 'DEPARTURE_TIME', 'DEPARTURE_DELAY', 'TAXI_OUT',
       'WHEELS_OFF', 'SCHEDULED_TIME', 'ELAPSED_TIME', 'AIR_TIME', 'DISTANCE',
       'WHEELS_ON', 'TAXI_IN', 'SCHEDULED_ARRIVAL', 'ARRIVAL_TIME',
       'ARRIVAL_DELAY', 'DIVERTED', 'CANCELLED', 'AIR_SYSTEM_DELAY',
       'SECURITY_DELAY', 'AIRLINE_DELAY', 'LATE_AIRCRAFT_DELAY',
       'WEATHER_DELAY', 'AIRLINE_NAME', 'ORIGIN_AIRPORT_NAME', 'ORIGIN_CITY',
       'ORIGIN_STATE', 'ORIGIN_COUNTRY', 'ORIGIN_LATITUDE', 'ORIGIN_LONGITUDE',
       'DEST_AIRPORT_NAME', 'DEST_CITY', 'DEST_STATE', 'DEST_COUNTRY',
       'DEST_LATITUDE', 'DEST_LONGITUDE', 'STATION_ORIGIN', 'DATE',
       'ORIGIN_AWND', 'ORIGIN_PRCP', 'ORIGIN_SNOW', 'ORIGIN_SNWD',
       'ORIGIN_TMAX', 'ORIGIN_TMIN', 'ORIGIN_WSF2', 'ORIGIN_WT01',
       'IS_DELAYED', 'ROUTE', 'TIME_OF_DAY', 'SEASON',
       'SCHEDULED

Selecionando features

In [20]:
cols = [
    'MONTH', 'DAY_OF_WEEK', 'SCHEDULED_DEPARTURE_HOUR', 'SCHEDULED_ARRIVAL_HOUR', 'SCHEDULED_TIME', 'TIME_OF_DAY', 
    'SEASON', 'DISTANCE', 'ORIGIN_AIRPORT_PROFILE', 'DESTINATION_AIRPORT_PROFILE', 'AIRLINE', 'AIRLINE_PROFILE', 'ROUTE_PROFILE',
    'ORIGIN_AWND', 'ORIGIN_PRCP', 'ORIGIN_SNOW', 'ORIGIN_SNWD', 'ORIGIN_TMAX', 'ORIGIN_TMIN', 'ORIGIN_WSF2', 
    'ORIGIN_WT01', 'IS_DELAYED'
]
df = df[cols]
df.head()

,MONTH,DAY_OF_WEEK,SCHEDULED_DEPARTURE_HOUR,SCHEDULED_ARRIVAL_HOUR,SCHEDULED_TIME,TIME_OF_DAY,SEASON,DISTANCE,ORIGIN_AIRPORT_PROFILE,DESTINATION_AIRPORT_PROFILE,AIRLINE,AIRLINE_PROFILE,ROUTE_PROFILE,ORIGIN_AWND,ORIGIN_PRCP,ORIGIN_SNOW,ORIGIN_SNWD,ORIGIN_TMAX,ORIGIN_TMIN,ORIGIN_WSF2,ORIGIN_WT01,IS_DELAYED
0,1,4,0,4,205.0,OVERNIGHT,SUMMER,1448,0,1,AS,0,1,0.0,0.0,0.0,10.2,0.6,-3.3,0.0,0.0,0
1,1,4,0,7,280.0,OVERNIGHT,SUMMER,2330,1,0,AA,2,2,2.3,0.0,0.0,0.0,13.9,2.2,5.4,0.0,0
2,1,4,0,8,286.0,OVERNIGHT,SUMMER,2296,1,1,US,0,0,3.8,0.0,0.0,0.0,12.8,4.4,8.9,0.0,0
3,1,4,0,8,285.0,OVERNIGHT,SUMMER,2342,1,1,AA,2,0,2.3,0.0,0.0,0.0,13.9,2.2,5.4,0.0,0
4,1,4,0,3,235.0,OVERNIGHT,SUMMER,1448,2,0,AS,0,1,1.2,0.0,0.0,0.0,5.6,-3.2,4.0,0.0,0


Obtendo correlações

In [21]:
df_corr = df.copy()

seasons = [['WINTER', 'SPRING', 'AUTUMN', 'SUMMER']]
tome_of_day = [['OVERNIGHT', 'MORNING', 'AFTERNOON', 'EVENING']]

encoded_seasons = OrdinalEncoder(categories=seasons)
encoded_time_or_day = OrdinalEncoder(categories=tome_of_day)

df_corr['SEASON_NUM'] = encoded_seasons.fit_transform(df_corr[['SEASON']])
df_corr['TIME_OF_DAY_NUM'] = encoded_time_or_day.fit_transform(df_corr[['TIME_OF_DAY']])

num_cols = [
    'MONTH', 'DAY_OF_WEEK', 'SCHEDULED_DEPARTURE_HOUR', 'SCHEDULED_ARRIVAL_HOUR', 'SCHEDULED_TIME',
    'TIME_OF_DAY_NUM', 'SEASON_NUM', 'DISTANCE', 'ORIGIN_AIRPORT_PROFILE', 'DESTINATION_AIRPORT_PROFILE', 'ROUTE_PROFILE',
    'ORIGIN_AWND', 'ORIGIN_PRCP', 'ORIGIN_SNOW', 'ORIGIN_SNWD', 'ORIGIN_TMAX', 'ORIGIN_TMIN', 'ORIGIN_WSF2', 'ORIGIN_WT01', 'IS_DELAYED'
]

df_corr = df_corr[num_cols].corr()

In [22]:
fig = px.imshow(
    df_corr,
    text_auto='.2f',
    aspect='auto',
    color_continuous_scale='RdBu_r',
    zmin=-1, 
    zmax=1,
    title='Matriz de Correlação',
    labels=dict(color='Correlação')
)
fig.update_layout(
    height=600,
    template='plotly_white'
)
fig.show()

Balanceando target

In [23]:
target = 'IS_DELAYED'

df_atrasados = df[df[target] == 1]
df_pontuais = df[df[target] == 0]

n_atrasados = len(df_atrasados)
df_pontuais_bal = df_pontuais.sample(n=n_atrasados, random_state=42)

df_balanced = pd.concat([df_atrasados, df_pontuais_bal])
df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

Normalizando features

In [24]:
cols_categoricas = ['SEASON', 'TIME_OF_DAY', 'AIRLINE']
cols_numericas = [
    'DISTANCE', 'MONTH', 'DAY_OF_WEEK', 'SCHEDULED_DEPARTURE_HOUR', 'SCHEDULED_ARRIVAL_HOUR', 'SCHEDULED_TIME', 'ORIGIN_AIRPORT_PROFILE', 'DESTINATION_AIRPORT_PROFILE', 'AIRLINE_PROFILE', 'ROUTE_PROFILE',
    'ORIGIN_AWND', 'ORIGIN_PRCP', 'ORIGIN_SNOW', 'ORIGIN_SNWD', 'ORIGIN_TMAX', 'ORIGIN_TMIN', 'ORIGIN_WSF2', 'ORIGIN_WT01'
]

df_normalized = df_balanced.copy()

scaler = StandardScaler()
df_normalized[cols_numericas] = scaler.fit_transform(df_normalized[cols_numericas])

df_normalized = pd.get_dummies(df_normalized, columns=cols_categoricas, dtype=int)

df_normalized.head()

,MONTH,DAY_OF_WEEK,SCHEDULED_DEPARTURE_HOUR,SCHEDULED_ARRIVAL_HOUR,SCHEDULED_TIME,DISTANCE,ORIGIN_AIRPORT_PROFILE,DESTINATION_AIRPORT_PROFILE,AIRLINE_PROFILE,ROUTE_PROFILE,ORIGIN_AWND,ORIGIN_PRCP,ORIGIN_SNOW,ORIGIN_SNWD,ORIGIN_TMAX,ORIGIN_TMIN,ORIGIN_WSF2,ORIGIN_WT01,IS_DELAYED,SEASON_AUTUMN,SEASON_SPRING,SEASON_SUMMER,SEASON_WINTER,TIME_OF_DAY_AFTERNOON,TIME_OF_DAY_EVENING,TIME_OF_DAY_MORNING,TIME_OF_DAY_OVERNIGHT,AIRLINE_AA,AIRLINE_AS,AIRLINE_B6,AIRLINE_DL,AIRLINE_EV,AIRLINE_F9,AIRLINE_HA,AIRLINE_MQ,AIRLINE_NK,AIRLINE_OO,AIRLINE_UA,AIRLINE_US,AIRLINE_VX,AIRLINE_WN
0,-0.654094,1.552578,-0.746963,-0.608701,-0.507027,-0.508744,-0.143055,-1.299499,-0.350184,-1.033565,0.522878,-0.201303,-0.060793,-0.151571,-0.140800,-0.435182,0.346505,-0.325723,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1
1,0.562626,1.552578,0.506520,0.361136,-0.896204,-0.972680,-0.143055,1.442162,0.865447,0.097844,-0.066481,-0.201303,-0.060793,-0.151571,1.995173,1.877892,-0.060175,-0.325723,1,0,0,0,1,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0
2,0.562626,1.552578,1.133261,1.524940,1.244269,1.138864,1.131569,-1.299499,-0.350184,1.229252,-1.068392,-0.201303,-0.060793,-0.151571,0.931770,0.640195,-1.029948,-0.325723,1,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1
3,0.258446,0.543861,-1.164791,-1.190604,-0.623780,-0.877351,-1.417679,1.442162,-1.565815,-1.033565,-1.422007,-0.201303,-0.060793,-0.151571,0.775927,0.356133,-1.029948,-0.325723,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0
4,-0.958274,-1.473572,0.506520,0.361136,-0.961067,-0.828097,1.131569,-1.299499,-0.350184,-1.033565,0.228198,-0.116392,-0.060793,-0.151571,-0.498323,-0.881564,0.909599,-0.325723,1,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1


Exportando features

In [25]:
df_normalized.to_csv('../data/curated/features.csv', index=False)
df_normalized.to_pickle('../data/curated/features.pkl')

In [26]:
df_normalized.shape

(1353870, 41)